In [1]:
import pandas as pd
import numpy as np

real_time_data = pd.read_csv("../fetched_gold_metal_economy_headlines.csv")

In [3]:
real_time_data.head()

,Date,Headline
0,2025-04-26,Is UK prime property making a comeback as a sa...
1,2025-04-27,Metals experienced a near across-the-board dec...
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...
3,2025-04-27,The safe haven asset investors are flocking to...
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?"


In [4]:
real_time_data["Date"].unique()

array(['2025-04-26', '2025-04-27', '2025-04-28'], dtype=object)

In [5]:
real_time_data.columns = ["Dates", "News"]

In [6]:
real_time_data.head()

,Dates,News
0,2025-04-26,Is UK prime property making a comeback as a sa...
1,2025-04-27,Metals experienced a near across-the-board dec...
2,2025-04-27,Does Wall Street's Sell-Off Have You Intereste...
3,2025-04-27,The safe haven asset investors are flocking to...
4,2025-04-27,"Should You Buy Bitcoin While It's Under $95,000?"


## 2. Sentiment

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
from tqdm import tqdm
from torch.nn.functional import softmax
import pandas as pd


def sentiment_analysis(df):
    """
    df -> This is a dataframe that consists of all the NY times headline that we have got
    """
    model_name = "ProsusAI/finbert"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

    # news_subset_df = df[["Dates", "News"]]

    # news_subset_df["Dates"] = pd.to_datetime(
    #     news_subset_df["Dates"], format="mixed", dayfirst=True, errors="coerce"
    # )

    grouped = df.groupby("Dates")["News"].apply(list)

    daily_sentiments = []

    for date, news_list in tqdm(grouped.items(), desc="Computing FinBERT sentiment"):
        logits_list = []

        for sentence in news_list:
            inputs = tokenizer(
                sentence, return_tensors="pt", truncation=True, padding=True
            )
            inputs = {k: v.to("cpu") for k, v in inputs.items()}
            model.to("cpu")
            with torch.no_grad():
                logits = model(**inputs).logits.squeeze()
            logits_list.append(logits)

        stacked_logits = torch.stack(logits_list)
        avg_logits = torch.mean(stacked_logits, dim=0)
        probabilities = softmax(avg_logits, dim=0)

        labels = model.config.id2label
        final_probs = {labels[i]: float(probabilities[i]) for i in range(len(labels))}
        final_sentiment = max(final_probs, key=final_probs.get)

        daily_sentiments.append(
            {
                "Date": date,
                "Positive": final_probs.get("positive", 0.0),
                "Neutral": final_probs.get("neutral", 0.0),
                "Negative": final_probs.get("negative", 0.0),
                "Final Sentiment": final_sentiment,
            }
        )

    sentiment_group_df = pd.DataFrame(daily_sentiments)

    sentiment_group_df["Date"] = pd.to_datetime(
        sentiment_group_df["Date"], dayfirst=True, errors="coerce"
    )
    sentiment_group_df = sentiment_group_df.sort_values("Date").reset_index(drop=True)

    sentiment_group_df["sentiment"] = (
        sentiment_group_df["Positive"] - sentiment_group_df["Negative"]
    )

    # Sentiment returned as a pos, neg, nuetral and overall sentiment
    return sentiment_group_df[["Date", "sentiment"]]

/Users/visheshgupta/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
real_time_data_sentiment = sentiment_analysis(real_time_data)

Device set to use mps:0
Computing FinBERT sentiment: 3it [00:00,  5.46it/s]
/var/folders/mc/2wjfdchj6vsffbrpfbfgqw4w0000gn/T/ipykernel_35821/2702280834.py:60: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  sentiment_group_df["Date"] = pd.to_datetime(


In [9]:
real_time_data_sentiment

,Date,sentiment
0,2025-04-26,0.395923
1,2025-04-27,0.050091
2,2025-04-28,0.101628


## 3. Clustering

In [ ]:
import pickle

with open("topic_model.pkl", "rb") as file:
    topic_model = pickle.load(file)

topic_group_map = pd.read_csv("../topic_group_map.csv")

In [11]:
import re

months = r"\b(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)\b"
directions = r"\b(up|down|higher|lower|rise|rises|fall|falls|gain|gains|loses|loss|rebound|slip|climb|surge|drop|drops|edged|edges|recover|recovery|recovers|flat)\b"
numbers = r"[\d\.,]+[%$]?|\d{1,3}(,\d{3})*(\.\d+)?|\d+"
symbols = r"\/oz|rs|bn|usd|\$|%|oz"

In [12]:
def classify_headline(row):
    cleaned = re.sub(months, "", row.lower())
    cleaned = re.sub(directions, "", cleaned)
    cleaned = re.sub(numbers, "", cleaned)
    cleaned = re.sub(symbols, "", cleaned)
    cleaned = re.sub(r"[^\w\s]", "", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Get topic and probability from the model
    topic, prob = topic_model.transform([cleaned])

    if topic == -1 or len(topic) == 0:
        return "noise"
    else:
        # Ensure topic is valid and exists in the map
        try:
            merged_group = topic_group_map.loc[
                topic_group_map["Original_Topic"] == topic[0], "Macro_Group"
            ].values[0]
            return merged_group
        except IndexError:
            return -1


# Apply classification to each row in the 'News' column
real_time_data["Category"] = real_time_data["News"].apply(classify_headline)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.94it/s]
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
Batches: 100%|██████████| 1/1 [00:00<00:00, 14.96it/s]


In [13]:
category_to_topic = {
    1: "topic_china_fed_and_gold_focus",
    2: "topic_comex_gold_closing_prices",
    3: "topic_early_gold_trading_activity",
    4: "topic_economic_data_and_gold_performance",
    5: "topic_global_cues_and_gold_stability",
    6: "topic_global_investment_and_rate_impact",
    7: "topic_gold_climbs_in_trading",
    8: "topic_gold_holds_as_data_releases",
    9: "topic_gold_market_turning_points",
    10: "topic_gold_near_monthly_lows",
    11: "topic_gold_price_support_levels",  # New topic added
    12: "topic_gold_prices_and_metal_shares",
    13: "topic_gold_settlement_price_correction",
    14: "topic_gold_spot_price_outlook",
    15: "topic_gold_trading_momentum_continues",
    16: "topic_market_rates_and_gold_demand",
    17: "topic_rate_hikes/cuts_price_impact",
    18: "topic_time_sensitive_prediction",
    -1: "topic_noise",  # Noise topic at the end
}

# Map Category to Topic
real_time_data["Topic"] = real_time_data["Category"].map(category_to_topic)

In [14]:
import pandas as pd

# --- Step 1: Parse dates safely ---
real_time_data["Dates"] = pd.to_datetime(
    real_time_data["Dates"], dayfirst=True, errors="coerce"
)

# --- Step 2: Drop rows with invalid dates ---
real_time_data = real_time_data.dropna(subset=["Dates"])

# --- Step 3: Convert to date only (no time component) ---
real_time_data["Dates"] = real_time_data["Dates"].dt.date

# --- Step 4: Group by Dates and Topic_Label, count headline frequency ---
daily_counts = (
    real_time_data.groupby(["Dates", "Topic"]).size().reset_index(name="count")
)

In [15]:
all_topics = category_to_topic.values()

In [16]:
unique_dates = daily_counts["Dates"].unique()

# Create a new DataFrame with all combinations of dates and topics
date_topic_combinations = pd.MultiIndex.from_product(
    [unique_dates, all_topics], names=["Dates", "Topic"]
).to_frame(index=False)

# Merge with original data to get counts
complete_df = pd.merge(
    date_topic_combinations, daily_counts, on=["Dates", "Topic"], how="outer"
).fillna(0)

# Convert count to integer
complete_df["count"] = complete_df["count"].astype(int)

In [17]:
import pandas as pd

# Assuming your DataFrame is named 'df'
pivoted_df = complete_df.pivot_table(
    index="Dates",
    columns="Topic",
    values="count",
    fill_value=0,  # Fill missing combinations with 0
).reset_index()

# If you want the topics as columns in a specific order:
topic_columns = [
    "topic_china_fed_and_gold_focus",
    "topic_comex_gold_closing_prices",
    "topic_early_gold_trading_activity",
    "topic_economic_data_and_gold_performance",
    "topic_global_cues_and_gold_stability",
    "topic_global_investment_and_rate_impact",
    "topic_gold_climbs_in_trading",
    "topic_gold_holds_as_data_releases",
    "topic_gold_market_turning_points",
    "topic_gold_near_monthly_lows",
    "topic_gold_price_support_levels",
    "topic_gold_prices_and_metal_shares",
    "topic_gold_settlement_price_correction",
    "topic_gold_spot_price_outlook",
    "topic_gold_trading_momentum_continues",
    "topic_market_rates_and_gold_demand",
    "topic_rate_hikes/cuts_price_impact",
    "topic_time_sensitive_prediction",
    "topic_noise",
]

# Reorder columns
pivoted_df = pivoted_df[["Dates"] + topic_columns]

In [18]:
pivoted_df

Topic,Dates,topic_china_fed_and_gold_focus,topic_comex_gold_closing_prices,topic_early_gold_trading_activity,topic_economic_data_and_gold_performance,topic_global_cues_and_gold_stability,topic_global_investment_and_rate_impact,topic_gold_climbs_in_trading,topic_gold_holds_as_data_releases,topic_gold_market_turning_points,topic_gold_near_monthly_lows,topic_gold_price_support_levels,topic_gold_prices_and_metal_shares,topic_gold_settlement_price_correction,topic_gold_spot_price_outlook,topic_gold_trading_momentum_continues,topic_market_rates_and_gold_demand,topic_rate_hikes/cuts_price_impact,topic_time_sensitive_prediction,topic_noise
0,2025-04-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2025-04-27,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
2,2025-04-28,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0


In [19]:
real_time_data_sentiment

,Date,sentiment
0,2025-04-26,0.395923
1,2025-04-27,0.050091
2,2025-04-28,0.101628


In [27]:
import yfinance as yf

ticker = "GLD"
gold_data = yf.download(
    ticker,
    start="2025-04-23",
    end=None,
    progress=False,
    auto_adjust=False,
    actions=True,
)
gold_data

Price,Adj Close,Capital Gains,Close,Dividends,High,Low,Open,Stock Splits,Volume
Ticker,GLD,GLD,GLD,GLD,GLD,GLD,GLD,GLD,GLD
Date,,,,,,,,,
2025-04-23,303.649994,0.0,303.649994,0.0,304.739990,300.589996,304.179993,0.0,25421500
2025-04-24,308.070007,0.0,308.070007,0.0,308.500000,304.709991,306.980011,0.0,10845000
2025-04-25,304.730011,0.0,304.730011,0.0,305.369995,301.010010,301.779999,0.0,10691600
2025-04-28,309.070007,0.0,309.070007,0.0,309.109985,302.970001,304.149994,0.0,9329500


In [21]:
gold_data.reset_index(inplace=True)
gold_data.columns = [col[0] for col in gold_data.columns]

gold_data["Date"] = pd.to_datetime(gold_data["Date"])

In [22]:
real_time_data_sentiment["Date"] = pd.to_datetime(real_time_data_sentiment["Date"])
pivoted_df["Dates"] = pd.to_datetime(pivoted_df["Dates"])

combined_df = real_time_data_sentiment.merge(
    pivoted_df, left_on="Date", right_on="Dates", how="inner"
).drop("Dates", axis=1)

In [23]:
combined_df["Open"] = gold_data["Open"]

In [ ]:
combined_df["pct_change"] = combined_df["Open"].pct_change()

In [ ]:
304.179993

In [ ]:
(306.980011 - 304.179993) / 304.179993

0.00920513532919953

In [ ]:
combined_df["pct_change"].iloc[0] = 0.00920513532919953

In [36]:
combined_df

,Date,sentiment,pct_change,topic_china_fed_and_gold_focus,topic_comex_gold_closing_prices,topic_early_gold_trading_activity,topic_economic_data_and_gold_performance,topic_global_cues_and_gold_stability,topic_global_investment_and_rate_impact,topic_gold_climbs_in_trading,...,topic_gold_near_monthly_lows,topic_gold_price_support_levels,topic_gold_prices_and_metal_shares,topic_gold_settlement_price_correction,topic_gold_spot_price_outlook,topic_gold_trading_momentum_continues,topic_market_rates_and_gold_demand,topic_rate_hikes/cuts_price_impact,topic_time_sensitive_prediction,topic_noise
0,2025-04-26,0.395923,0.009205,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2025-04-27,0.050091,-0.016939,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
2,2025-04-28,0.101628,0.007853,0.0,0.0,2.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0


In [35]:
combined_df = combined_df[
    [
        "Date",
        "sentiment",
        "pct_change",
        "topic_china_fed_and_gold_focus",
        "topic_comex_gold_closing_prices",
        "topic_early_gold_trading_activity",
        "topic_economic_data_and_gold_performance",
        "topic_global_cues_and_gold_stability",
        "topic_global_investment_and_rate_impact",
        "topic_gold_climbs_in_trading",
        "topic_gold_holds_as_data_releases",
        "topic_gold_market_turning_points",
        "topic_gold_near_monthly_lows",
        "topic_gold_price_support_levels",
        "topic_gold_prices_and_metal_shares",
        "topic_gold_settlement_price_correction",
        "topic_gold_spot_price_outlook",
        "topic_gold_trading_momentum_continues",
        "topic_market_rates_and_gold_demand",
        "topic_rate_hikes/cuts_price_impact",
        "topic_time_sensitive_prediction",
        "topic_noise",
    ]
]

In [25]:
import pickle

with open("final_combined_model.pkl", "rb") as file:
    final_combined_model = pickle.load(file)

In [37]:
import numpy as np


def df_to_windowed_df(dataframe, n=3):
    dates = []
    X, Y = [], []

    for i in range(n, len(dataframe)):
        window = dataframe.iloc[i - n : i + 1]  # n previous + 1 target row

        if len(window) != n + 1:
            continue  # skip if not enough data

        values = window[
            [
                "pct_change",
                "sentiment",
                "topic_china_fed_and_gold_focus",
                "topic_comex_gold_closing_prices",
                "topic_early_gold_trading_activity",
                "topic_economic_data_and_gold_performance",
                "topic_global_cues_and_gold_stability",
                "topic_global_investment_and_rate_impact",
                "topic_gold_climbs_in_trading",
                "topic_gold_holds_as_data_releases",
                "topic_gold_market_turning_points",
                "topic_gold_near_monthly_lows",
                "topic_gold_price_support_levels",
                "topic_gold_prices_and_metal_shares",
                "topic_gold_settlement_price_correction",
                "topic_gold_spot_price_outlook",
                "topic_gold_trading_momentum_continues",
                "topic_market_rates_and_gold_demand",
                "topic_rate_hikes/cuts_price_impact",
                "topic_time_sensitive_prediction",
                "topic_noise",
            ]
        ].to_numpy()
        x, y = values[:-1], values[-1][0]

        dates.append(dataframe.index[i])  # the date corresponding to the target
        X.append(x)
        Y.append(y)

    ret_df = pd.DataFrame({"Target Date": dates})

    X = np.array(X)
    for i in range(n):
        ret_df[f"Target-{n - i}"] = list(
            X[:, i]
        )  # each element is a pair: [Close, sentiment]

    ret_df["Target"] = Y

    return ret_df

In [38]:
windowed_df = df_to_windowed_df(combined_df, n=2)
windowed_df

,Target Date,Target-2,Target-1,Target
0,2,"[0.00920513532919953, 0.3959233742207289, 0.0,...","[-0.0169392534397389, 0.05009138584136963, 0.0...",0.007853


In [39]:
def windowed_df_to_date_X_y(windowed_dataframe):
    dates = windowed_dataframe["Target Date"].to_numpy()

    feature_cols = windowed_dataframe.columns[1:-1]

    X = []
    for _, row in windowed_dataframe[feature_cols].iterrows():
        # Each val is a [Close, Sentiment] pair
        values = [val for val in row]  # Keep as 2D structure
        X.append(values)

    X = np.array(X)  # Shape will be (num_rows, n, 2)
    Y = windowed_dataframe["Target"].to_numpy()

    return dates, X.astype(np.float32), Y.astype(np.float32)


dates, X, y = windowed_df_to_date_X_y(windowed_df)

In [40]:
X

array([[[ 0.00920514,  0.39592338,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          1.        ],
        [-0.01693925,  0.05009139,  0.        ,  0.        ,
          3.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  1.        ,  1.        ,  0.        ,
          0.        ,  0.        ,  0.        ,  0.        ,
          2.        ]]], dtype=float32)

In [69]:
final_combined_model.predict(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step


array([[181.61235]], dtype=float32)